# Razorpay AI Buildathon 2026 — Recovery Intelligence

## Problem

When a payment fails, the naive move is "retry everything" or "contact
everyone." Some customers recover on their own. Some respond strongly to
a payment link, some to a retry, some need escalation, some should be
left alone.

**Core thesis: recovery rate is the wrong optimization target. Incremental
contribution is.**

The system answers, for every failed payment:

> *Should we intervene? If yes, with what action? Is that action
> economically worthwhile?*

Output: **ACT / WAIT / DON'T ACT**, with the recommended action, estimated
recovery probability, estimated incremental lift vs. WAIT, estimated
incremental contribution, confidence, and an explanation.

## Architecture

```
Failed Payment → Context Builder → Recovery Intelligence (this notebook)
→ Score Candidate Actions → WAIT Baseline → Incremental Recovery Estimation
→ Economic Allocation → ACT / WAIT / DON'T ACT → Deterministic Safety/Policy
→ Execution Agent → Payment Link / Retry / Reminder / Alternate Method / Human
→ Outcome → Experimentation + Evaluation → Model Feedback
```

The ML model in this notebook provides **intelligence only**
(`P(recovery | state, action)`). A separate, deterministic decision engine
(`ml/decision_engine.py`) turns that into the final action — the model
never executes anything directly.

## Dataset provenance

Source: `Niyaz05/AI-Revenue-Recovery-Orchestrator` (public repo), synthetic
recovery episodes. **This is not real Razorpay customer data.**
`provenance` = `SYNTHETIC_TRAINING_DATA` (train/val) /
`HELD_OUT_OFFLINE_EVAL` (test), confirmed directly from the files below.


## Section 2 — Environment and Imports

In [ ]:

import sys, os, json, time
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ml import config, data_loader, features, model as model_mod
from ml import action_scorer, economics, decision_engine, evaluation, predictor

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)
np.random.seed(config.RANDOM_SEED)
print("Environment ready.")


## Section 3 — Dataset Loading

Loads `episodes_train.parquet`, `episodes_val.parquet`, `episodes_test.parquet`
as the **official** split (per project spec — we do not re-split by row).
`data_loader.load_and_verify` also asserts zero `episode_id` overlap across
splits and identical schemas, and raises if either check fails.

In [ ]:

frames = data_loader.load_and_verify()
train, val, test = frames['train'], frames['val'], frames['test']
print(f"train: {train.shape}, val: {val.shape}, test: {test.shape}")
overlap = data_loader.verify_episode_overlap(frames)
print("Episode overlap (train/val, train/test, val/test):", overlap)
assert overlap.is_clean
print("PASS: zero episode_id leakage across splits.")


## Section 4 — Dataset Audit

Shape, dtypes, missingness, target/action distributions, episode structure.

In [ ]:

print("dtypes:\n", train.dtypes)
print("\nMissing values (train):", train.isna().sum().sum())
print("\nTarget (resolved) distribution (train):")
print(train['resolved'].value_counts(normalize=True))
print("\nAction distribution (train):")
print(train['action'].value_counts(normalize=True).round(4))


In [ ]:

ep_len = train.groupby('episode_id').size()
print("Episode length distribution (train):")
print(ep_len.value_counts().sort_index())
print("\nUnique episodes: train", train.episode_id.nunique(),
      "val", val.episode_id.nunique(), "test", test.episode_id.nunique())
print("\nTimestep distribution (train):")
print(train['timestep'].value_counts().sort_index())


## Section 5 — Temporal Leakage Audit

**Method:** we inspected the raw Parquet files directly (not just the
metadata report) by tracing full multi-step episodes end to end. Findings
below are stated as **ASSUMPTION → OBSERVED → REQUIRED CHANGE** wherever
the spec's assumption needed correcting.

### Finding 1 — row = one (state, action) transition; `resolved` is the outcome of THAT action
Each row is `(episode_id, timestep)`. `action` is the action chosen at that
state. `resolved`/`reward`/`recovered_amount`/`done` are the **result of
executing that action** — available only after the action, never before.

Verified: `resolved=True` occurs **only** on the last row of an episode
(0 occurrences on any non-terminal row), and `done=True` marks episode
termination (either because it resolved, or `STOP_RECOVERY` was chosen, or
the 7-step horizon was reached — 12,467 terminal-but-unresolved rows in
train).

- **ASSUMPTION** (spec §5): "resolved... corresponds closely to
  recovered_amount > 0... inspect whether it's a valid transition outcome."
- **OBSERVED**: `resolved` is *exactly* consistent with
  `recovered_amount > 0` (0 mismatches either direction, checked
  exhaustively on train), and is generated as the direct outcome of the
  action taken at that timestep — not a delayed/future-episode aggregate.
- **REQUIRED CHANGE**: none. `resolved` is used as-is as the supervised
  target. No leakage.

### Finding 2 — `reward` / `episode_return` / `episode_length` / `done` are post-action and excluded
`episode_length` and `episode_return` are only knowable once the full
episode has played out (aggregates over all timesteps) — pure future
leakage if used as features. `reward` and `done` are per-row but are the
*result*, not the *input*, of the action. All four are excluded from
`ml/features.py` (see `config.EXCLUDED_COLUMNS`).

### Finding 3 — `behavior_prob` carries no information
`behavior_prob` is a **constant 1.0** in every row of every split. It is
not a usable logging-policy propensity score, so no valid inverse-propensity
correction is possible with this field. This reinforces the causal caveat
in Section 7 of the spec: incremental-lift estimates here are
**model-estimated, not causally identified**.

### Finding 4 — action costs in the raw `reward` column do NOT match the assumed flat `ACTION_COSTS` table
- **ASSUMPTION** (spec §13): flat costs, e.g. WAIT=0, RETRY=5, ...,
  ESCALATE_TO_HUMAN=150.
- **OBSERVED**: back-solving `recovered_amount - reward` on resolved rows
  gives highly variable, state-dependent implied costs (e.g.
  ESCALATE_TO_HUMAN averages ~243 with std ~114, not a flat 150; WAIT
  itself shows non-zero, sometimes strongly negative reward even when
  unresolved). The dataset's `reward` field does not decompose into a
  clean `amount − fixed_cost` formula.
- **REQUIRED CHANGE**: we do **not** use the raw `reward`/`episode_return`
  columns for economics at all (they're excluded as leakage anyway). The
  `ACTION_COSTS` dict in `ml/config.py` is kept exactly as specified —
  it's an explicit, separately-configurable economic **assumption**, not
  something we claim to have derived from this field. This is flagged here
  so it can be recalibrated against real operational costs later without
  touching the model.

### Finding 5 — `allowed_actions` is a decision-time policy mask, not a feature
It shrinks predictably with `attempt_count`/`timestep` (e.g. `RETRY` drops
out after attempt 2; the action set collapses to
`ESCALATE_TO_HUMAN, STOP_RECOVERY, WAIT` at high timesteps). It's real,
pre-action information, but a *constraint on candidate actions*, not a
predictive feature — `ml/action_scorer.py` uses it to restrict which
candidate actions get scored per state; it is not fed to CatBoost.

### Final decision-time feature set
See `config.NUMERICAL_FEATURES` / `BOOLEAN_FEATURES` / `CATEGORICAL_FEATURES`
/ `DERIVED_FEATURES` below — every one of these is knowable *before* the
action for that row is taken.

In [ ]:

print("Excluded (post-outcome / ID / metadata) columns:")
print(config.EXCLUDED_COLUMNS)
print("\nDecision-time model features:")
print(config.ALL_MODEL_FEATURES)

# Verify Finding 1 exhaustively
resolved_true = train['resolved'] == True
last_rows = train.sort_values(['episode_id','timestep']).groupby('episode_id').tail(1)
non_last_resolved = train.merge(
    last_rows[['episode_id','timestep']], on=['episode_id','timestep'],
    how='left', indicator=True
)
non_last_resolved_true = ((non_last_resolved['_merge']=='left_only') & (non_last_resolved['resolved']==True)).sum()
print("\nresolved=True on a non-terminal row (should be 0):", non_last_resolved_true)
print("recovered_amount>0 but resolved=False (should be 0):", ((train.recovered_amount>0)&(train.resolved==False)).sum())
print("recovered_amount==0 but resolved=True (should be 0):", ((train.recovered_amount==0)&(train.resolved==True)).sum())

# Verify Finding 4
cost_check = train.copy()
cost_check['implied_cost'] = cost_check['recovered_amount'] - cost_check['reward']
print("\nImplied action cost from raw reward (resolved rows only) vs. assumed ACTION_COSTS:")
implied = cost_check[cost_check.resolved].groupby('action')['implied_cost'].agg(['mean','std'])
implied['assumed_cost'] = implied.index.map(config.ACTION_COSTS)
print(implied)


## Section 6 — Feature Engineering

Only decision-time-safe features (`ml/features.py`). Derived features:
`amount_to_ltv_ratio`, `attempt_bucket`, `failure_recency_bucket`,
`intervention_pressure` — all computed purely from pre-action columns.

In [ ]:

X_train = features.prepare_features(train)
y_train = features.extract_target(train)
X_val = features.prepare_features(val)
y_val = features.extract_target(val)
X_test = features.prepare_features(test)
y_test = features.extract_target(test)

print("Feature matrix shape:", X_train.shape)
print("\nCategorical features (native CatBoost handling):", features.get_categorical_feature_names())
X_train.head(3)


## Section 7 — Model Training

One action-conditioned `CatBoostClassifier`: `P(resolved | state, action)`.
Config per spec §8 (`Logloss`, `AUC`, 500 iterations, depth 7, lr 0.05,
seed 42). Early stopping/best-model selection uses the **validation** set
only; the model is never fit on validation.

In [ ]:

t0 = time.time()
clf = model_mod.train_model(X_train, y_train, X_val, y_val, params=config.CATBOOST_PARAMS)
print(f"Training done in {time.time()-t0:.1f}s. Best iteration: {clf.get_best_iteration()}")


## Section 8 — Model Evaluation

ROC-AUC / PR-AUC / accuracy / precision / recall / F1 / Brier, on val and test. Accuracy is reported but is **not** the primary success metric — see Section 16.

In [ ]:

val_proba = model_mod.predict_proba(clf, X_val)
test_proba = model_mod.predict_proba(clf, X_test)

val_metrics = evaluation.model_metrics(y_val, val_proba)
test_metrics = evaluation.model_metrics(y_test, test_proba)

metrics_df = pd.DataFrame([val_metrics, test_metrics], index=['validation','test'])
metrics_df.round(4)


In [ ]:

# Calibration
calib = evaluation.calibration_table(y_test, test_proba, n_bins=10)
fig, ax = plt.subplots(figsize=(5,5))
ax.plot(calib['mean_predicted'], calib['mean_observed'], marker='o', label='Model (test)')
ax.plot([0,1],[0,1],'--',color='gray',label='Perfect calibration')
ax.set_xlabel('Mean predicted probability'); ax.set_ylabel('Mean observed recovery rate')
ax.set_title('Calibration curve — test set'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('outputs/calibration_curve.png', dpi=110); plt.show()


## Section 9 — Action Scoring

For a sample of held-out decision points (test set, deduplicated to the
**first decision point per episode** — the actual failed-payment moment),
generate one candidate row per allowed action and score
`P(recovery | state, action)` with the single trained model.

In [ ]:

# first decision point (timestep 0) per episode in test = the actual failed-payment moment
test_first = test[test['timestep']==0].reset_index(drop=True)
print("Test failed-payment decision points (timestep 0):", test_first.shape)

# states = features only, action will be re-generated per candidate
state_cols = [c for c in test_first.columns if c != 'action']
sample_states = test_first[state_cols].copy()

scored = action_scorer.score_candidate_actions(clf, sample_states)
scored = action_scorer.attach_wait_baseline(scored)
print("Scored candidate rows:", scored.shape)
scored[['_state_row_id','action','recovery_probability','wait_probability']].head(10)


## Section 10 — Incremental Opportunity

`incremental_lift(action) = P(recovery|action) − P(recovery|WAIT)`

In [ ]:

scored = economics.add_incremental_lift(scored)
print(scored.groupby('action')['incremental_lift'].describe().round(4))


## Section 11 — Economic Allocation

`incremental_value = incremental_lift × amount`
`net_incremental_contribution = incremental_value − action_cost − friction_cost`

Costs from `config.ACTION_COSTS` (kept separate from the model — see
Section 5, Finding 4). Friction is a simple, transparent function of
**prior** intervention count (pre-action information only).

In [ ]:

scored = economics.add_economic_value(scored)
scored.groupby('action')[['action_cost','friction_cost','incremental_value','net_incremental_contribution']].mean().round(2)


## Section 12 — Decision Engine

Deterministic: for every state, remove policy-invalid actions, keep
candidates clearing both the minimum lift and minimum net-contribution
thresholds, pick the best by net contribution; if nothing clears the bar,
fall back to WAIT / DON'T ACT.

In [ ]:

decisions = decision_engine.decide(scored)
print(decisions['decision'].value_counts())
print("\nRecommended action distribution when ACT:")
print(decisions.loc[decisions['decision']=='ACT','recommended_action'].value_counts())
decisions.head(5)


## Section 13 — Baseline Comparison

Six policies evaluated on the same scored candidates: No Action, Blind
Retry, Rule-Based, Propensity (highest raw recovery probability), Economic
(highest gross value after cost, ignoring WAIT), and our Incremental-Lift
policy.

In [ ]:

policy_names = ['no_action','blind_retry','rule_based','propensity','economic','incremental']
comparison = evaluation.compare_policies(scored, policy_names)
comparison.round(2)


## Section 14 — Visualizations

In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# recovery probability by action
order = config.ALL_ACTIONS
scored.boxplot(column='recovery_probability', by='action', ax=axes[0,0], rot=45)
axes[0,0].set_title('Recovery probability by action'); axes[0,0].set_xlabel(''); axes[0,0].set_ylabel('P(recovery)')
plt.suptitle('')

# incremental lift distribution
scored[scored['action']!='WAIT']['incremental_lift'].hist(bins=40, ax=axes[0,1], color='#4C72B0')
axes[0,1].set_title('Incremental lift distribution (non-WAIT actions)')
axes[0,1].set_xlabel('Incremental lift vs WAIT'); axes[0,1].set_ylabel('Count')

# net contribution by policy
axes[1,0].bar(comparison['policy'], comparison['net_incremental_contribution'], color='#55A868')
axes[1,0].set_title('Net incremental contribution by policy'); axes[1,0].tick_params(axis='x', rotation=45)
axes[1,0].set_ylabel('Net incremental contribution (currency units)')

# intervention rate by policy
axes[1,1].bar(comparison['policy'], comparison['intervention_rate'], color='#C44E52')
axes[1,1].set_title('Intervention rate by policy'); axes[1,1].tick_params(axis='x', rotation=45)
axes[1,1].set_ylabel('Fraction of states intervened on')

plt.tight_layout()
os.makedirs('outputs', exist_ok=True)
plt.savefig('outputs/policy_visualizations.png', dpi=110)
plt.show()


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12,4.5))
decisions['decision'].value_counts().plot(kind='bar', ax=axes[0], color='#8172B2')
axes[0].set_title('Decision engine outcome distribution'); axes[0].tick_params(axis='x', rotation=0)

act_only = decisions[decisions['decision']=='ACT']
act_only['recommended_action'].value_counts().plot(kind='bar', ax=axes[1], color='#64B5CD')
axes[1].set_title('Recommended action distribution (ACT decisions only)'); axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.savefig('outputs/decision_distributions.png', dpi=110); plt.show()


## Section 15 — Example Decisions

Four illustrative cases: high natural recovery → DON'T ACT / WAIT; low
natural recovery + strong action → ACT; best action differs from the
highest raw recovery probability (our policy vs. propensity policy); and
an expensive intervention the policy rejects on economics.

In [ ]:

def explain(state_row_id):
    d = decisions[decisions['_state_row_id']==state_row_id].iloc[0]
    print(f"Decision: {d['decision']}")
    print(f"Recommended action: {d['recommended_action']}")
    print(f"  WAIT probability: {d['baseline_recovery_probability']:.3f}")
    print(f"  Action probability: {d['action_recovery_probability']:.3f}")
    print(f"  Incremental lift: {d['estimated_incremental_lift']*100:+.1f} pp")
    print(f"  Incremental value: {d['estimated_incremental_value']:.1f}")
    print(f"  Action cost: {d['action_cost']:.1f}")
    print(f"  Net contribution: {d['estimated_net_contribution']:.1f}")
    print(f"  Reason: {d['reason']}")
    print()

# Case A: high natural (WAIT) recovery probability -> should be DONT_ACT/WAIT
high_wait = decisions.sort_values('baseline_recovery_probability', ascending=False)
candidates_a = high_wait[high_wait['decision']!='ACT']
print("=== Case A: High natural recovery -> DON'T ACT / WAIT ===")
if len(candidates_a):
    explain(candidates_a.iloc[0]['_state_row_id'])
else:
    print("(No DONT_ACT/WAIT cases in this sample; showing lowest-lift ACT instead)")
    explain(decisions.sort_values('estimated_incremental_lift').iloc[0]['_state_row_id'])

# Case B: low natural recovery + strong action -> ACT
print("=== Case B: Low natural recovery, strong action -> ACT ===")
low_wait_act = decisions[decisions['decision']=='ACT'].sort_values('baseline_recovery_probability')
explain(low_wait_act.iloc[0]['_state_row_id'])

# Case C: best action differs from highest raw propensity
print("=== Case C: Incremental policy vs. propensity policy can disagree ===")
prop = evaluation.apply_baseline_policy(scored, 'propensity')
inc = evaluation.apply_baseline_policy(scored, 'incremental')
merged_pc = prop.merge(inc, on='_state_row_id', suffixes=('_propensity','_incremental'))
disagree = merged_pc[merged_pc['chosen_action_propensity'] != merged_pc['chosen_action_incremental']]
print(f"{len(disagree)} / {len(merged_pc)} states where propensity and incremental policies disagree.")
if len(disagree):
    row = disagree.iloc[0]
    print(f"  State {row['_state_row_id']}: propensity picks {row['chosen_action_propensity']} "
          f"(P={row['recovery_probability_propensity']:.3f}), incremental policy picks "
          f"{row['chosen_action_incremental']} (P={row['recovery_probability_incremental']:.3f}, "
          f"lift={row['incremental_lift_incremental']*100:+.1f}pp vs propensity lift="
          f"{row['incremental_lift_propensity']*100:+.1f}pp)")
print()

# Case D: expensive intervention rejected on economics
print("=== Case D: Expensive intervention the policy rejects ===")
escalate_candidates = scored[(scored['action']=='ESCALATE_TO_HUMAN')]
rejected = escalate_candidates[escalate_candidates['net_incremental_contribution'] < config.MIN_NET_CONTRIBUTION_TO_ACT]
if len(rejected):
    sid = rejected.iloc[0]['_state_row_id']
    explain(sid)
else:
    print("(No ESCALATE_TO_HUMAN candidates fell below threshold in this sample.)")


## Section 16 — Final Results

In [ ]:

summary = f'''
MODEL PERFORMANCE (test set)
  ROC-AUC:  {test_metrics['roc_auc']:.4f}
  PR-AUC:   {test_metrics['pr_auc']:.4f}
  Accuracy: {test_metrics['accuracy']:.4f}  (secondary metric only)
  F1:       {test_metrics['f1']:.4f}
  Brier:    {test_metrics['brier_score']:.4f}

POLICY PERFORMANCE (sample of {len(sample_states)} test failed-payment states)
{comparison.set_index('policy')[['intervention_rate','net_incremental_contribution']].round(2).to_string()}

Our incremental-lift policy vs. No Action baseline:
  net incremental contribution gain = {comparison.set_index('policy').loc['incremental','net_incremental_contribution'] - comparison.set_index('policy').loc['no_action','net_incremental_contribution']:.2f}

LIMITATIONS
  - All data is synthetic (Niyaz05/AI-Revenue-Recovery-Orchestrator generator).
    Do not present these numbers as real Razorpay recovery performance.
  - behavior_prob is constant -> no valid propensity weighting; incremental
    lift is MODEL-ESTIMATED, not a causally identified treatment effect.
  - ACTION_COSTS are an explicit economic assumption (see Section 5,
    Finding 4), not derived from the dataset's own reward field.
  - Baselines compare using the trained model's own probability estimates
    for counterfactual actions, since the logged data observed only one
    action per state (standard limitation of off-policy evaluation on
    logged/observational data).
'''
print(summary)
with open('outputs/final_results_summary.txt','w') as f:
    f.write(summary)


## Section 17 — Export

Export the trained CatBoost model, feature configuration, action cost
configuration, and decision engine configuration so they can be loaded by
the production modules (`ml/predictor.py`) without retraining.

In [ ]:

os.makedirs('outputs', exist_ok=True)
model_mod.save_model(clf, 'outputs/recovery_model.cbm')

export_config = {
    'numerical_features': config.NUMERICAL_FEATURES,
    'boolean_features': config.BOOLEAN_FEATURES,
    'categorical_features': config.CATEGORICAL_FEATURES,
    'derived_features': config.DERIVED_FEATURES,
    'action_costs': config.ACTION_COSTS,
    'friction_cost_per_prior_intervention': config.FRICTION_COST_PER_PRIOR_INTERVENTION,
    'min_net_contribution_to_act': config.MIN_NET_CONTRIBUTION_TO_ACT,
    'min_incremental_lift_to_act': config.MIN_INCREMENTAL_LIFT_TO_ACT,
    'catboost_params': config.CATBOOST_PARAMS,
    'test_metrics': test_metrics,
    'val_metrics': val_metrics,
}
with open('outputs/model_config_export.json','w') as f:
    json.dump(export_config, f, indent=2, default=str)

# small worked example using the full predictor pipeline, output schema per spec section 24
example_decisions = predictor.predict_decisions(clf, sample_states.head(5))
example_schema = predictor.to_output_schema(example_decisions)
with open('outputs/example_decisions.json','w') as f:
    json.dump(example_schema, f, indent=2, default=str)

print("Exported:")
print(" - outputs/recovery_model.cbm")
print(" - outputs/model_config_export.json")
print(" - outputs/example_decisions.json")
print(" - outputs/calibration_curve.png")
print(" - outputs/policy_visualizations.png")
print(" - outputs/decision_distributions.png")
print(" - outputs/final_results_summary.txt")
